In [9]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully!")

# diagnostik
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

# tampilan
sns.set_style("whitegrid")
%matplotlib inline

print("Libraries loaded successfully!")


Libraries loaded successfully!
Libraries loaded successfully!


In [10]:
# =========================
# Imports
# =========================
import numpy as np
import pandas as pd
import statsmodels.api as sm

# =========================
# Data generation
# =========================
np.random.seed(42)

n = 5000  # number of households

df = pd.DataFrame({
    "household_id": range(1, n + 1),
    "village": np.random.choice(["Village A", "Village B", "Village C"], n),
    "monthly_income": np.random.normal(5_000_000, 2_000_000, n).clip(900_000),
    "house_condition": np.random.choice(
        ["poor", "average", "good"], n, p=[0.45, 0.35, 0.20]
    ),
    "num_dependents": np.random.randint(0, 6, n)
})

print("Total rows:", len(df))

df.to_csv("dummy_social_assistance_targeting.csv", index=False)

# =========================
# Eligibility (ground truth)
# =========================
df["eligible_actual"] = (
    (df["monthly_income"] < 3_500_000) &
    (df["house_condition"] == "poor") &
    (df["num_dependents"] >= 2)
).astype(int)

# =========================
# Logistic Regression (Full Model)
# =========================

# Convert categorical variable to dummy variables
df["income_million"] = df["monthly_income"] / 1_000_000

df_model = pd.get_dummies(df, columns=["house_condition"])

print(df_model.columns)

# Features
X = df_model[
    ["income_million",
     "num_dependents",
     "house_condition_average",
     "house_condition_good"]
]

# Add intercept
X = sm.add_constant(X)

# Target
y = df_model["eligible_actual"]
X = X.astype(float)
y = y.astype(float)


# Train model
logit_model = sm.Logit(y, X).fit(disp=False)

print(logit_model.summary())


# =========================
# Simulate targeting errors
# =========================
df["received_assistance"] = np.where(
    df["eligible_actual"] == 1,
    np.random.choice([1, 0], n, p=[0.7, 0.3]),   # exclusion error
    np.random.choice([0, 1], n, p=[0.85, 0.15])  # inclusion error
)

df["targeting_status"] = np.select(
    [
        (df["eligible_actual"] == 1) & (df["received_assistance"] == 1),
        (df["eligible_actual"] == 1) & (df["received_assistance"] == 0),
        (df["eligible_actual"] == 0) & (df["received_assistance"] == 1)
    ],
    ["accurate", "exclusion_error", "inclusion_error"],
    default="accurate"
)

print(df.head())
print("\nTargeting summary:")
print(df["targeting_status"].value_counts())

# Export the updated 5000-row dataframe to your CSV file

Total rows: 5000
Index(['household_id', 'village', 'monthly_income', 'num_dependents',
       'eligible_actual', 'income_million', 'house_condition_average',
       'house_condition_good', 'house_condition_poor'],
      dtype='object')
                           Logit Regression Results                           
Dep. Variable:        eligible_actual   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4995
Method:                           MLE   Df Model:                            4
Date:                Sun, 15 Feb 2026   Pseudo R-squ.:                  0.7637
Time:                        12:17:00   Log-Likelihood:                -275.94
converged:                      False   LL-Null:                       -1167.6
Covariance Type:            nonrobust   LLR p-value:                     0.000
                              coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------

c:\Users\nanni\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [11]:
# 1 = mis-targeted, 0 = correctly targeted
df["mis_targeted"] = df["targeting_status"].apply(
    lambda x: 1 if x in ["inclusion_error", "exclusion_error"] else 0
)

df["mis_targeted"].value_counts()


mis_targeted
0    4223
1     777
Name: count, dtype: int64

In [12]:
df.groupby("mis_targeted")[["monthly_income", "num_dependents"]].mean()
pd.crosstab(df["house_condition"], df["mis_targeted"], normalize="index")



mis_targeted,0,1
house_condition,,
average,0.866933,0.133067
good,0.852674,0.147326
poor,0.823738,0.176262


In [13]:
import statsmodels.api as sm

X = df[["monthly_income", "num_dependents",  ]]
X = sm.add_constant(X)

y = df["mis_targeted"]


logit_model = sm.Logit(y, X).fit()
logit_model.summary()


Optimization terminated successfully.
         Current function value: 0.431254
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           mis_targeted   No. Observations:                 5000
Model:                          Logit   Df Residuals:                     4997
Method:                           MLE   Df Model:                            2
Date:                Sun, 15 Feb 2026   Pseudo R-squ.:                0.001641
Time:                        12:17:00   Log-Likelihood:                -2156.3
converged:                       True   LL-Null:                       -2159.8
Covariance Type:            nonrobust   LLR p-value:                   0.02889
==================================================================================
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -1.4825      0.119    -12.480      0.000      -1.715      -1.250
monthly_income -5.129e-08   2.01e-08     -2.557      0.011   -9.06e-08    -1.2e-08
num_dependents     0.0177      0.023      0.780      0.436      -0.027       0.062
==================================================================================
"""

In [ ]:
import joblib
joblib.dump(logit_model, "logit_model_bansos.pkl")


['logit_model_bansos.pkl']

: 

## Logistic Regression Interpretation

The dependent variable is **mis_targeted**, indicating whether a household
experienced mistargeting in social assistance distribution.

Key findings:
- **Monthly income** has a negative coefficient, suggesting higher income
  households tend to have lower probability of mistargeting, although the
  effect is not statistically significant.
- **Number of dependents** has a positive coefficient, indicating households
  with more dependents are more likely to experience mistargeting.
- The model converges successfully, indicating stable estimation.
